In [0]:
%pip install yfinance

In [0]:
%run ./00_config

In [0]:
import json
import time
import math
from datetime import datetime
import yfinance as yf

In [0]:
import pandas as pd


assets = pd.read_csv(f"{RAW}/assets/assets.csv")
TICKERS = assets["ticker"].tolist()
ticker_map = dict(zip(assets["ticker"], assets["asset_id"]))

print("Ticker mapping from assets.csv:", ticker_map)

In [0]:
while True:
    live_ticks = []
    # Fetch 1-minute interval data for latest tick
    df_data = yf.download(TICKERS, period="1d", interval="1m")["Close"].iloc[-1]

    for ticker in TICKERS:
        price = df_data[ticker] if len(TICKERS) > 1 else df_data
        if not math.isnan(price):
            live_ticks.append({
                "asset_id": ticker_map[ticker],
                "price_date": str(datetime.now().date()),
                "open_price": float(price),
                "close_price": float(price),
                "adjusted_close": float(price),
                "volume": 0,
                "created_at": str(datetime.now()),
            })

    # Dump tick array to landing volume folder as JSON
    file_path = f"{RAW}/ticks/ticks_{datetime.now():%Y%m%d_%H%M%S}.json"
    dbutils.fs.put(file_path, "\n".join(json.dumps(t) for t in live_ticks), overwrite=False)
    print(f"wrote {len(live_ticks)} ticks to {file_path}")
    time.sleep(60)  # Wait 1 minute before next tick